# 00 · Cloud setup and full run (Colab / Kaggle)

One notebook that takes a cloud runtime from an empty kernel to a saved research run:

1. set the writable directories **before** `pavana_ml` is imported,
2. install the code bundle from Hugging Face at a pinned revision and verify every file,
3. download the curated data snapshot at its own pinned revision,
4. run the same three stages as notebooks 02 → 03 → 04 (audit/align, features, experiment).

What the numbers are, and are not:

- The target is the **CAMS model-field PM2.5 at t+6 h** on one grid cell. It is **not** observed
  station PM2.5, and nothing here is station validation or proof of operational forecast skill.
- Weather is **ERA5 reanalysis**: a valid timestamp does not prove the data were available when a
  forecast would have been issued.
- Hotspot rows are **satellite detections, not confirmed fires**; a zero count means no record in
  the cached window, not that no fire burned.
- The snapshot has no `issued_at`, so the CAMS issue-time baseline cannot be run at all.
- Artifacts stay `research_only: true`, `operational_ready: false`, `observation_validated: false`.

This notebook is written for a **fresh runtime**. It shares kernel variables between cells, but every
stage reads its input from disk, so a single stage can be re-run without redoing the earlier ones.


In [ ]:
"""Cloud runner: pins, writable directories, and the switch for a smoke run."""

import json
import os
import sys
from pathlib import Path

REPO_ID = "myudak/pavana"

# Two separate pins. Never `main`: a later upload must not change what a recorded result was
# computed from. Data PR #5 was merged on 2026-10-15 and `main` carries the same bytes (verified
# against this manifest), but the experiment keeps the commit its CP1-CP3 evidence was hashed on.
DATA_REF = "main"
DATA_REVISION = "aaa0d2714102b7d3a7b3ad0beb2b1da1ce6a6dbe"
CODE_REF = "refs/pr/6"
CODE_REVISION = "a9a0d6d98008e08e80d6e978d82cfc89f79282af"  # commit that carries this bundle
RESOLVE_CODE_HEAD = False  # True resolves CODE_REF once and installs that commit instead

RUN_VERSION = "cams-t6h-retro-kaggle-v1"
SMOKE = False  # True: 30 rounds. An execution check, never a reported result.
MAX_ROUNDS = 30 if SMOKE else 500
RUN_VERSION_EFFECTIVE = f"{RUN_VERSION}-smoke" if SMOKE else RUN_VERSION


def cloud_work_root() -> Path:
    """A writable directory: /kaggle/working, /content, or an override for local checks."""
    override = os.environ.get("PAVANA_CLOUD_WORK_ROOT")
    if override:
        return Path(override).expanduser()
    for candidate in ("/kaggle/working", "/content"):
        if Path(candidate).is_dir() and os.access(candidate, os.W_OK):
            return Path(candidate) / "pavana"
    return Path.cwd() / "cloud-run"


WORK_ROOT = cloud_work_root()
DATA_DIR = WORK_ROOT / "data"
ARTIFACTS_DIR = WORK_ROOT / "artifacts"
CODE_ROOT = WORK_ROOT / "code-download"
for path in (DATA_DIR, ARTIFACTS_DIR, CODE_ROOT):
    path.mkdir(parents=True, exist_ok=True)

# `pavana_ml.paths` reads these when it is imported, so they must be set before the first import.
os.environ["PAVANA_ML_DATA_DIR"] = str(DATA_DIR)
os.environ["PAVANA_ML_ARTIFACTS_DIR"] = str(ARTIFACTS_DIR)

already = sorted(name for name in sys.modules if name.split(".")[0] == "pavana_ml")
print("python         :", sys.version.split()[0])
print("work root      :", WORK_ROOT, "| writable:", os.access(WORK_ROOT, os.W_OK))
print("data dir       :", DATA_DIR)
print("artifacts dir  :", ARTIFACTS_DIR)
print("pavana_ml already imported in this kernel:", already or "no")
if already:
    print()
    print("RESTART THE KERNEL before continuing: the directories above are read when `pavana_ml`")
    print("is imported, so a stale DATA_DIR/ARTIFACTS_DIR would write outside this runtime.")

## What has and has not been verified

- The modules and notebooks 02–04 were verified locally (CP1–CP3). The directory override, the code
  bundle and this all-in-one flow were then checked in an **isolated environment** on the author's
  machine: a fresh virtualenv, a scratch data root and the same pinned revisions.
- That check reproduced the CP1 and CP2 artifact hashes byte-for-byte (`2e8505f0…4461` for the
  aligned hours, `e210ca6a…ba16` for the model table) and the CP3 holdout metrics exactly. Compare
  the row counts and hashes printed below with that evidence.
- **The first real cloud run has since been done** (Kaggle kernel `muhamadkemalfaza/pavana`,
  2026-10-15, run `cams-t6h-retro-kaggle-v1`). Its 6,882 test predictions are bit-identical to the
  local run, RMSE matches exactly, MAE differs by at most 1 ulp, and only `upwind_score` moved
  (3.6e-16 relative, float summation order). Colab has not been run.
- Colab and any other runtime remain unverified: treat the next new environment as an execution
  check before quoting a metric from it.
- A `table_sha256` that differs from the CP2 evidence is not automatically a data difference: with
  pandas 2.2 the same table is written with `datetime64[ns]` instead of `[us]`, so the bytes change
  while the values, the row count and the NaN counts stay identical. Compare values when the
  runtime's pandas build differs from the one the hash was recorded with.
- Internet must stay **on** for the install and the snapshot download. No GPU, and no provider API
  key, is needed: the snapshot is public.

Next cells: set the pins and the writable root.


In [ ]:
import importlib.util
import subprocess


def pip(*args: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", *args], check=True)


if importlib.util.find_spec("huggingface_hub") is None:
    print("huggingface_hub is missing; installing it from PyPI")
    pip("install", "--quiet", "huggingface-hub>=0.26")

from huggingface_hub import HfApi  # noqa: E402

api = HfApi()
for label, ref in (("code", CODE_REF), ("data", DATA_REF)):
    info = api.dataset_info(REPO_ID, revision=ref)
    print(f"{label} ref  :", ref, "->", info.sha)
print("Internet works and the dataset is readable. No GPU and no provider API key is needed.")

In [ ]:
import hashlib

from huggingface_hub import snapshot_download


def file_sha256(path: Path) -> str:
    """Hash the downloaded bytes with the same function CODE_MANIFEST.json was built with."""
    return hashlib.sha256(path.read_bytes()).hexdigest()


if RESOLVE_CODE_HEAD:
    CODE_REVISION = HfApi().dataset_info(REPO_ID, revision=CODE_REF).sha
    print("resolved", CODE_REF, "once for this run ->", CODE_REVISION)
else:
    print("pinned code revision ->", CODE_REVISION)

download = Path(
    snapshot_download(
        REPO_ID,
        repo_type="dataset",
        revision=CODE_REVISION,
        allow_patterns=["code/**"],
        local_dir=CODE_ROOT,
    )
)
bundle = download / "code"
manifest = json.loads((bundle / "CODE_MANIFEST.json").read_text(encoding="utf-8"))

mismatched = []
for entry in manifest["files"]:
    path = bundle / entry["path"].removeprefix("code/")
    if not path.is_file() or file_sha256(path) != entry["sha256"]:
        mismatched.append(entry["path"])
assert not mismatched, f"bundle does not match CODE_MANIFEST.json: {mismatched}"

print("bundle root :", bundle)
print(f"verified {len(manifest['files'])} files against CODE_MANIFEST.json at {CODE_REVISION[:12]}")
print("The data revision is a separate pin:", DATA_REVISION)

In [ ]:
pip("install", "--quiet", "--upgrade", f"{bundle / 'ml'}[hub]")

import pavana_ml  # noqa: E402
from pavana_ml.paths import ARTIFACTS_DIR as RESOLVED_ARTIFACTS  # noqa: E402
from pavana_ml.paths import DATA_DIR as RESOLVED_DATA  # noqa: E402

print("package       :", Path(pavana_ml.__file__).parent)
print("DATA_DIR      :", RESOLVED_DATA)
print("ARTIFACTS_DIR :", RESOLVED_ARTIFACTS)
assert Path(RESOLVED_DATA) == DATA_DIR, RESOLVED_DATA
assert Path(RESOLVED_ARTIFACTS) == ARTIFACTS_DIR, RESOLVED_ARTIFACTS
print("The installed package resolves the writable cloud directories, not repository defaults.")

## Stages

Each cell below is a stage of the pipeline. The logic lives in `pavana_ml`, not in this notebook, so
the same rules that notebooks 02–04 explain are the ones running here.

| Stage | Module | Writes |
| --- | --- | --- |
| Audit and align | `pavana_ml.data.research` | `data/interim/hourly_aligned.parquet`, `audit_report.json` |
| Features | `pavana_ml.features.model_table` | `data/processed/model_table.parquet`, `model_table.metadata.json` |
| Experiment | `pavana_ml.models.experiment` | `artifacts/<version>/` (report, metrics, predictions, C/D/E models) |


In [ ]:
from pavana_ml.data.research import MANIFEST_PATHS, download_snapshot

curated = sorted(MANIFEST_PATHS.values())
receipt_path = DATA_DIR / "dataset_snapshot.json"
missing = [path for path in curated if not (DATA_DIR / path).exists()]
if missing or not receipt_path.exists():
    print(f"downloading the {len(curated)} curated files at {DATA_REVISION[:12]}")
    print("the raw FIRMS batches stay on the Hub: the model table only reads the curated parquet")
    download_snapshot(DATA_DIR, DATA_REVISION, include=curated)
else:
    print("curated snapshot already present in this runtime; nothing re-downloaded")

stored = json.loads(receipt_path.read_text(encoding="utf-8"))
print("requested     :", stored["requested_revision"])
print("resolved      :", stored["resolved_revision"])
print("manifest sha  :", stored["manifest_sha256"])
print("verified files:", len(stored["verified_files"]))
assert stored["resolved_revision"] == DATA_REVISION, stored["resolved_revision"]
print("Every downloaded file matched the SHA256 recorded in the manifest at that commit.")

In [ ]:
from pavana_ml.data.research import prepare_cached_data, save_prepared

hourly, hotspots, report = prepare_cached_data(DATA_DIR)
audit = save_prepared(hourly, report, DATA_DIR)
snapshot = report["snapshot"]

print("aligned rows   :", f"{report['aligned']['rows']:,}")
print("hotspot rows   :", f"{len(hotspots):,}  (detections, not fire events)")
print("period (UTC)   :", report["aligned"]["start_utc"], "->", report["aligned"]["end_utc"])
print("unique hours   :", report["aligned"]["unique_timestamps"])
print("CAMS NaN hours :", report["aligned"]["missing_hour_rows"])
print("snapshot       :", snapshot["revision"], "| verified:", snapshot["revision_verified"])
print("evidence       :", snapshot["revision_source"] or snapshot["reason"])
print("interim sha256 :", audit["hourly_sha256"])
present_na = {name: count for name, count in report["missing_by_column"].items() if count}
print("columns with missing values (kept as NaN):", present_na or "none")
assert report["aligned"]["rows"] == 24_432, report["aligned"]["rows"]
print()
print("Limitations recorded by the audit:")
for line in report["limitations"]:
    print("-", line)

In [ ]:
from pavana_ml.features.model_table import build_model_table, load_feature_inputs, save_model_table

hourly, hotspots, inputs = load_feature_inputs(DATA_DIR)
table, metadata = build_model_table(hourly, hotspots, inputs=inputs)
table_saved = save_model_table(table, metadata, DATA_DIR)
eligibility = metadata["eligibility"]

print("rows / columns:", f"{len(table):,} / {len(table.columns)}")
print("target        :", metadata["target"], f"(kind: {metadata['target_kind']})")
print("eligible rows :", f"{eligibility['eligible']:,}")
print("excluded      :", eligibility["excluded_missing_target"], "without a target at t+6,")
print("               ", eligibility["excluded_incomplete_pm25_history"], "with incomplete history")
print("table sha256  :", table_saved["table_sha256"])
nan_in_eligible = {c: n for c, n in metadata["feature_nan_counts_in_eligible"].items() if n}
print("NaN in eligible feature rows (never filled):", nan_in_eligible or "none")
print()
print("Limitations recorded with the table:")
for line in metadata["limitations"]:
    print("-", line)

In [ ]:
from pavana_ml.models.experiment import (
    artifact_status,
    chronological_split,
    load_model_table,
    run_experiment,
)

table, metadata = load_model_table(DATA_DIR)
_, windows = chronological_split(table, metadata)
print("splits        :", {name: window["rows"] for name, window in windows["splits"].items()})
print("dropped in gap:", windows["dropped_in_gap"], "| ineligible:", windows["ineligible_rows"])

try:
    run = run_experiment(
        table,
        metadata,
        max_rounds=MAX_ROUNDS,
        version=RUN_VERSION_EFFECTIVE,
        if_exists="reuse",
    )
except ValueError as error:
    print("reuse refused:", error)
    print("Set RUN_VERSION above to a new name (for example -v2) and run this cell again.")
    raise

results, report = run["results"], run["report"]
test_rows = results[(results["split"] == "test") & (results["period"] == "all")]
print("run directory :", run["output_dir"])
print("selected      :", report["selected_variant"], "on", report["selected_on"])
print("validation MAE:", report["validation_mae"])
print()
print("Holdout metrics; every variant scores the same eligible test rows:")
print(test_rows[["variant", "features_n", "n", "mae", "rmse"]].to_string(index=False))
print()
print(report["status"]["statement"])
for entry in artifact_status(ARTIFACTS_DIR):
    print(f"  {entry['version']}: evaluated={entry['evaluated']} app_ready={entry['app_ready']}")

In [ ]:
import importlib.metadata
import platform
import shutil
from datetime import UTC, datetime


def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


cloud_run = {
    "recorded_at_utc": datetime.now(UTC).isoformat(timespec="seconds"),
    "notebook": "00_cloud_setup.ipynb",
    "code": {"ref": CODE_REF, "revision": CODE_REVISION, "resolved_from_head": RESOLVE_CODE_HEAD},
    "data": {
        "requested_revision": stored["requested_revision"],
        "resolved_revision": stored["resolved_revision"],
        "manifest_sha256": stored["manifest_sha256"],
    },
    "run": {
        "version": report["version"],
        "directory": str(run["output_dir"]),
        "max_rounds": MAX_ROUNDS,
        "smoke": SMOKE,
    },
    "runtime": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "cpu_count": os.cpu_count(),
    },
    "packages": {
        name: package_version(name)
        for name in (
            "pavana-ml",
            "pandas",
            "numpy",
            "pyarrow",
            "lightgbm",
            "scikit-learn",
            "huggingface-hub",
        )
    },
    "status": report["status"],
}
cloud_path = WORK_ROOT / "cloud_run.json"
cloud_path.write_text(json.dumps(cloud_run, indent=2) + "\n", encoding="utf-8")
archive = shutil.make_archive(
    str(WORK_ROOT / f"pavana-{report['version']}"), "zip", run["output_dir"]
)

print(json.dumps(cloud_run, indent=2))
print()
print("cloud_run.json:", cloud_path)
print("artifact zip  :", archive)
print("Download both before the session ends: the working directory does not survive it.")

## Before you close the session

- `/kaggle/working` (and Colab's `/content`) is **wiped when the runtime ends**. Download
  `cloud_run.json` and the zip archive printed by the last cell, or copy them to Drive.
- Record both revisions: the **code** revision that was installed and the **data** revision. They
  are separate pins and a report is only reproducible with both.
- A model fitted in a session cannot be handed to the app: `evaluated` is not
  `operational_ready`, and this target is a model field, not an observation.
- Kaggle caps session time and may interrupt a long run; the experiment itself takes seconds on CPU
  after the data is in place, so the download is what usually dominates.
- A few `fatal: not a git repository` lines during training are expected: `git_commit()` asks the
  working directory for a commit, a cloud runtime is not a checkout, and the run records `null`.
